# Route II evaluation with the shared Route I/II/III/IV framework

This notebook evaluates an existing folder of Route II predicted MIDIs with the same metric set used in Route I:
- velocity MAE
- BSSL / BSTL Pearson correlation
- BSSL / BSTL MAE
- BSSL / BSTL cosine similarity
- BSSL / BSTL Spearman rank correlation

You can reuse the same notebook later for Route III / IV by only changing the prediction folder and the label.


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "score_hpt":
    REPO_ROOT = REPO_ROOT.parent
elif not (REPO_ROOT / "score_hpt").exists():
    raise RuntimeError("Please open this notebook from the repository root or from repo_root/score_hpt.")

SCORE_HPT_DIR = REPO_ROOT / "score_hpt"
PYTORCH_DIR = SCORE_HPT_DIR / "pytorch"
DATA_ANALYSIS_SRC = REPO_ROOT / "data_analysis" / "src"
for path in [PYTORCH_DIR, DATA_ANALYSIS_SRC]:
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

print("REPO_ROOT:", REPO_ROOT)
print("PYTORCH_DIR:", PYTORCH_DIR)
print("DATA_ANALYSIS_SRC:", DATA_ANALYSIS_SRC)


## Parameters

`PRED_MIDI_DIR` should point to the folder of predicted MIDIs exported by `score_hpt/pytorch/inference.py --mode dataset` or by any later Route III / IV exporter.


In [ ]:
DATASET_TYPE = "smd"                  # "smd" | "maestro" | "maps"
DATASET_DIR = REPO_ROOT / "PATH_TO_DATASET"
PRED_MIDI_DIR = REPO_ROOT / "PATH_TO_ROUTE2_PRED_MIDIS"
ROUTE_LABEL = "route2_veloest"
INSTRUMENT_PATH = REPO_ROOT / "PATH_TO_SOUNDFONT.sf2"   # replace with a real .sf2/.sfz file
OUT_DIR = SCORE_HPT_DIR / "outputs" / "route2_eval" / DATASET_TYPE

SPLIT = "test"
MAPS_PIANOS = "both"
OVERWRITE = False

RENDER_SR = 44100
EVAL_SR = 22050
FPS = 50.0
FFT_SIZE = 1024
BSSL_MODE = "sone"
NUM_SAMPLES = 2048
NORMALIZATION = "zscore"
BACKEND = "auto"
DEVICE = None

# Optional helper only for printing a command template.
CHECKPOINT_PATH = None
INFERENCE_EXTRA_ARGS = ""


In [ ]:
if CHECKPOINT_PATH:
    cmd = (
        f"cd {SCORE_HPT_DIR} && "
        f"python pytorch/inference.py --mode dataset "
        f"--checkpoint-path {CHECKPOINT_PATH} "
        f"{INFERENCE_EXTRA_ARGS}"
    )
    print("Optional route2 export command:\n")
    print(cmd)
else:
    print("Set PRED_MIDI_DIR to an existing prediction folder, or fill CHECKPOINT_PATH and run the command manually.")


In [ ]:
from direct_invension.eval_framework import (
    build_dataset_prediction_manifest,
    evaluate_prediction_manifest,
    evaluation_results_to_dataframe,
)

manifest_path = OUT_DIR / f"{ROUTE_LABEL}_manifest.json"
route2_manifest = build_dataset_prediction_manifest(
    dataset_type=DATASET_TYPE,
    dataset_dir=DATASET_DIR,
    pred_midi_dir=PRED_MIDI_DIR,
    label=ROUTE_LABEL,
    split=SPLIT,
    maps_pianos=MAPS_PIANOS,
    manifest_out=manifest_path,
)
route2_manifest["label"], len(route2_manifest["items"]), len(route2_manifest["missing_gt_midis"])


In [ ]:
route2_eval = evaluate_prediction_manifest(
    route2_manifest,
    instrument_path=INSTRUMENT_PATH,
    out_dir=OUT_DIR / "evaluation",
    render_sr=RENDER_SR,
    eval_sr=EVAL_SR,
    frames_per_second=FPS,
    fft_size=FFT_SIZE,
    bssl_mode=BSSL_MODE,
    num_samples=NUM_SAMPLES,
    normalization=NORMALIZATION,
    device=DEVICE,
    backend=BACKEND,
    skip_existing_render=not OVERWRITE,
)
route2_eval["summary"]


In [ ]:
summary_df = evaluation_results_to_dataframe(route2_eval)
display(summary_df)
summary_df


## Reuse for Route III / IV

For Route III / IV, keep this notebook unchanged and only switch:
- `PRED_MIDI_DIR`
- `ROUTE_LABEL`
- `INSTRUMENT_PATH` if the renderer changes

The manifest builder and evaluator stay the same.
